In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, random_split
from models.lstm_model import LSTM_Model
from utils import EarlyStopping, get_device, EpochTrainer, build_seq

from sklearn.preprocessing import MinMaxScaler

# Parameters

In [3]:
batch_size = 32
epochs = 30
seq_len = 20
hidden_size = 64
num_output = 1
num_layers = 2
dropout = 0.2
bidirectional = False
patience = 5
test_size = 0.2
lr = 0.001

In [4]:
input_cols = ["Open", "High", "Low", "Close", "Volume"]
target_cols = ['Close']
ticker = "VOO"

In [5]:
model_output_filename = "lstm_checkpoint"

# Get Sample Data

## Load data from yfinance

In [6]:
import yfinance as yf
import pandas as pd
import numpy as np

In [7]:
df = yf.download(ticker , start = "2020-01-01" , end = "2026-04-25")
data = df.dropna().copy()

[*********************100%***********************]  1 of 1 completed


# Data Processing

## Scale inputs and target

In [8]:
x_scaler = MinMaxScaler()
y_scaler = MinMaxScaler()

X_all = x_scaler.fit_transform(data[input_cols].values)
y_all = y_scaler.fit_transform(data[target_cols].values)

## Build Sequence

In [9]:
X_seq , y_seq = build_seq( X_all , y_seq , seq_len )

## Convert to Tensor

In [10]:
X_tensor = torch.tensor(X_seq, dtype=torch.float32)
y_tensor = torch.tensor(y_seq, dtype=torch.float32)

In [11]:
full_dataset = TensorDataset(X_tensor, y_tensor)

## Split data into training and testing

In [12]:
input_size = X_tensor.shape[-1]

In [13]:
test_records = int(test_size * len(full_dataset))
train_records = len(full_dataset) - test_records

train_dataset, test_dataset = random_split(
    full_dataset,
    [train_records, test_records],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


# Prepare Model

## Prepare config of the model

In [14]:
preprocess_config = {
    "seq_length": seq_len,
    "input_size": input_size,
    "task_type": "classification",
}

In [15]:
model_config = {
    "input_size": input_size,
    "hidden_size": hidden_size,
    "num_output": num_output,
    "num_layers": num_layers,
    "dropout": dropout,
    "bidirectional": bidirectional,
}

In [16]:
device = get_device()

In [17]:
model = LSTM_Model(
    input_size=input_size,
    hidden_size=hidden_size,
    num_output=num_output,
    num_layers=num_layers,
    dropout=dropout,
    bidirectional=bidirectional,
).to(device)

In [18]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.MSELoss()

In [19]:
early_stopping = EarlyStopping(
    patience=patience,
    path=f"../checkpoints/{model_output_filename}.pt",
    checkpoint_data={
        "model_config": model_config,
        "preprocess_config": preprocess_config,
        "target_cols": target_cols,
        "input_cols": input_cols,
        "x_scaler": x_scaler,
        "y_scaler": y_scaler,

    },
)

# Loop epochs and train model

In [20]:
epoch_trainer = EpochTrainer(
    model = model,
    early_stopping = early_stopping,
    device = device,
    optimizer = optimizer,
    criterion = criterion,
    eval_method = 'RMSE'
)

In [21]:
for epoch in range(epochs):

    avg_train_loss, avg_val_loss, result = epoch_trainer(train_loader , test_loader )

    for key, value in result.items():        
        print(f"Epoch {epoch + 1:3d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | {key}: {value:.4f}")
    
    # ==================== Early Stopping Check ====================

    early_stopping(avg_val_loss, model)

    if early_stopping.early_stop:
        
        print("Early stopping triggered! Training stopped.")
        
        break
        

Epoch   1 | Train Loss: 0.0823 | Val Loss: 0.0110 | RMSE: 0.1047
Epoch   2 | Train Loss: 0.0055 | Val Loss: 0.0014 | RMSE: 0.0372
Epoch   3 | Train Loss: 0.0033 | Val Loss: 0.0009 | RMSE: 0.0300
Epoch   4 | Train Loss: 0.0027 | Val Loss: 0.0006 | RMSE: 0.0243
Epoch   5 | Train Loss: 0.0027 | Val Loss: 0.0011 | RMSE: 0.0335
Epoch   6 | Train Loss: 0.0025 | Val Loss: 0.0006 | RMSE: 0.0252
Epoch   7 | Train Loss: 0.0025 | Val Loss: 0.0005 | RMSE: 0.0224
Epoch   8 | Train Loss: 0.0024 | Val Loss: 0.0004 | RMSE: 0.0210
Epoch   9 | Train Loss: 0.0022 | Val Loss: 0.0005 | RMSE: 0.0221
Epoch  10 | Train Loss: 0.0022 | Val Loss: 0.0008 | RMSE: 0.0278
Epoch  11 | Train Loss: 0.0018 | Val Loss: 0.0004 | RMSE: 0.0191
Epoch  12 | Train Loss: 0.0018 | Val Loss: 0.0009 | RMSE: 0.0304
Epoch  13 | Train Loss: 0.0021 | Val Loss: 0.0011 | RMSE: 0.0337
Epoch  14 | Train Loss: 0.0021 | Val Loss: 0.0011 | RMSE: 0.0334
Epoch  15 | Train Loss: 0.0018 | Val Loss: 0.0005 | RMSE: 0.0230
Epoch  16 | Train Loss: 0

In [22]:
# for epoch in range(epochs):

#     # ----------------------------- Train -----------------------------
    
#     model.train()
#     train_loss = 0.0

#     for batch_x, batch_y in train_loader:
#         batch_x = batch_x.to(device)
#         batch_y = batch_y.to(device)

#         optimizer.zero_grad()
#         output = model(batch_x)
#         loss = criterion(output, batch_y)

#         loss.backward()
#         optimizer.step()

#         train_loss += loss.item()

#     avg_train_loss = train_loss / len(train_loader)

#     # ------------------- Validation -------------------
    
#     model.eval()
#     val_loss = 0.0
#     correct = 0
#     total = 0

#     with torch.no_grad():
#         for batch_x, batch_y in test_loader:
#             batch_x = batch_x.to(device)
#             batch_y = batch_y.to(device)

#             output = model(batch_x)
#             loss = criterion(output, batch_y)
#             val_loss += loss.item()

#     avg_val_loss = val_loss / len(test_loader)

#     # For regression: calculate RMSE (Root Mean Squared Error)
#     rmse = np.sqrt(avg_val_loss)

#     print(
#         f"Epoch {epoch + 1:3d} | "
#         f"Train Loss: {avg_train_loss:.4f} | "
#         f"Val Loss: {avg_val_loss:.4f} | "
#         f"RMSE: {rmse:.4f}"
#     )

#     # ==================== Early Stopping Check ====================
    
#     early_stopping(avg_val_loss, model)

#     if early_stopping.early_stop:
        
#         print("Early stopping triggered! Training stopped.")
        
#         break